# Imvo project: all GPU work on Colab

**Use this in the browser** (colab.research.google.com): downloads do not work from the VS Code Colab extension.

1. **Runtime → Change runtime type → T4 GPU**.
2. Run all cells. Cell 2 asks for `colab/colab_bundle.zip` (make it with `python src/make_colab_bundle.py`).
3. The last cell downloads `embeddings_all.zip` (and `results_all.zip` if you ran the analysis cell). Unzip both into the project folder.

Total GPU time on a T4: ~40 min for LaBSE (4 spelling variants) + ~15 min for the second encoder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
up = files.upload()  # choose colab_bundle.zip
!unzip -o -q colab_bundle.zip && ls src data/processed
!ls labels 2>/dev/null || echo "no labels bundled (provisional threshold will be used)"

In [ ]:
!pip -q install sentence-transformers pyarrow tabulate
# regenerate the modernised variants from the shipped candidate table (deterministic)
!for r in all asp prefix; do python src/normalise.py apply --rules $r; done

## LaBSE: raw + three spelling variants
Modern variants reuse raw embeddings for unchanged chunks (all English text).

In [ ]:
!python src/embed.py --variant raw --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_asp --chunks data/processed/chunks_modern_asp.parquet --reuse raw --batch 128 --shard 8192
!python src/embed.py --variant modern_prefix --chunks data/processed/chunks_modern_prefix.parquet --reuse raw --batch 128 --shard 8192

## Second encoder (robustness): multilingual-e5-base, raw + modern

In [ ]:
E5 = 'intfloat/multilingual-e5-base'
!python src/embed.py --variant raw --model {E5} --prefix 'query: ' --batch 128 --shard 8192
!python src/embed.py --variant modern --chunks data/processed/chunks_modern.parquet --model {E5} --prefix 'query: ' --reuse raw --batch 128 --shard 8192

In [ ]:
import numpy as np, glob
for f in sorted(glob.glob('data/embeddings/*.npy')):
    e = np.load(f); n = np.linalg.norm(e.astype('float32'), axis=1)
    print(f, e.shape, 'norms', n.min().round(3), n.max().round(3))

## Optional: run the analysis here too
Labels are optional: if `labels/pairs_labelled.csv` is in the bundle it is used; otherwise the provisional `THRESHOLD` below is used. The E5 run analyses raw + modern only (the variants embedded above). Provisional E5 results are rough: E5 similarities run higher than LaBSE, so 0.55 admits more false pairs.

In [ ]:
THRESHOLD = 0.55   # provisional; ignored if labels/pairs_labelled.csv was bundled
!python src/run_all.py analyse --threshold {THRESHOLD}
!python src/run_all.py analyse --model-tag multilingual-e5-base --threshold {THRESHOLD}

In [ ]:
!cd data/embeddings && zip -q -0 ../../embeddings_all.zip *.npy *_ids.txt
!zip -q -r results_all.zip results 2>/dev/null; ls -la *.zip
files.download('embeddings_all.zip')
import os
if os.path.exists('results_all.zip'): files.download('results_all.zip')